# PR Conflict Audit Notebook

Un notebook pentru a crea un Pull Request și a verifica dacă apar conflicte folosind GitPython și GitHub API.

In [3]:
# 1. Import Required Libraries
import os
import requests
from git import Repo

In [4]:
# 2. Crearea unui Pull Request (PR) folosind GitPython
# Setează variabilele de repo și branch
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
REPO_OWNER = 'adrianenc11-hue'
REPO_NAME = 'kelionai-v2'
BRANCH_NAME = 'chore/ai-cleanup'
BASE_BRANCH = 'master'

# Creează PR folosind GitHub API
pr_url = f'https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/pulls'
headers = {'Authorization': f'token {GITHUB_TOKEN}', 'Accept': 'application/vnd.github.v3+json'}
data = {
    'title': 'chore: remove all Gemini/OpenAI, Claude+ElevenLabs only, doc+test+config cleanup',
    'head': BRANCH_NAME,
    'base': BASE_BRANCH,
    'body': 'Eliminare completă Gemini/OpenAI, doar Claude+ElevenLabs, cleanup doc/test/config.'
}
response = requests.post(pr_url, headers=headers, json=data)
if response.status_code == 201:
    print('PR creat cu succes:', response.json()['html_url'])
else:
    print('Eroare la crearea PR:', response.text)

Eroare la crearea PR: {
  "message": "Bad credentials",
  "documentation_url": "https://docs.github.com/rest",
  "status": "401"
}


In [5]:
# 3. Listarea PR-urilor deschise
list_pr_url = f'https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/pulls'
list_response = requests.get(list_pr_url, headers=headers)
open_prs = list_response.json() if list_response.status_code == 200 else []
for pr in open_prs:
    print(f"PR #{pr['number']}: {pr['title']} - {pr['html_url']}")

In [6]:
# 4. Verificarea conflictelor pentru PR-uri
conflict_results = []
for pr in open_prs:
    pr_details_url = f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/pulls/{pr['number']}"
    pr_details = requests.get(pr_details_url, headers=headers).json()
    has_conflict = pr_details.get('mergeable') is False
    conflict_results.append({
        'number': pr['number'],
        'title': pr['title'],
        'url': pr['html_url'],
        'has_conflict': has_conflict
    })

In [ ]:
# mai clar5. Afișarea rezultatelor conflictelor
import pandas as pd
from IPython.display import display
if 'conflict_results' in globals():
    if conflict_results:
        df = pd.DataFrame(conflict_results)
        display(df[['number', 'title', 'url', 'has_conflict']])
    else:
        print('Nu există PR-uri deschise.')
else:
    print('conflict_results nu este definit. Rulează toate celulele anterioare în ordine.')

Nu există PR-uri deschise.


In [ ]:
# 6. Automatizare: rulează toate celulele notebook-ului
from IPython.display import display, Markdown
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor
import os
notebook_path = os.path.abspath('c:\\Users\\adria\\kelionai-v2\\kelionai-v2\\.github\\pr_conflict_audit.ipynb')
with open(notebook_path) as f:
    nb = nbformat.read(f, as_version=4)
ep = ExecutePreprocessor(timeout=600, kernel_name='python3')
try:
    ep.preprocess(nb, {'metadata': {'path': os.path.dirname(notebook_path)}})
    display(Markdown('**Toate celulele au fost rulate cu succes!**'))
except Exception as e:
    display(Markdown(f'**Eroare la rularea automată:** {e}'))

In [8]:
# Verificare dacă GITHUB_TOKEN este setat în mediu
import os
print("GITHUB_TOKEN:", "setat" if os.getenv("GITHUB_TOKEN") else "NU este setat")

GITHUB_TOKEN: NU este setat
